# FireRedTTS3 — دبلجة عربية واستنساخ صوت

هذا الدفتر هو المسار الأخف من Fish S2 Pro. يدعم FireRedTTS3 العربية واستنساخ الصوت من مرجع صوتي. استخدم Colab مع **L4 أو A100 GPU**، وليس TPU، ثم شغّل الخلايا بالترتيب.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'
import re
import subprocess
import torch

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True,
    text=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('لم يتم العثور على GPU. اختر GPU من Runtime > Change runtime type، وليس TPU.')
print(gpu.stdout.strip())
match = re.search(r'([0-9]+) MiB', gpu.stdout)
if match and int(match.group(1)) < 14000:
    raise RuntimeError('ذاكرة GPU أقل من 14GB. استخدم T4 أو L4 أو A100، وليس TPU.')
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch لا يرى CUDA. أعد تشغيل جلسة Colab بعد اختيار GPU.')
print('CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))


## تثبيت workspace والاعتمادات

نثبت الاعتمادات اللازمة فقط لمسار FireRedTTS3-Base، من دون FlashAttention الإجباري.


In [ ]:
%cd /content
!apt-get update -qq && apt-get install -y -qq ffmpeg libsox-dev
!pip -q install --upgrade pip
!rm -rf /content/dub22
!git clone --depth 1 https://github.com/dhiyaddineb-hue/dub22.git /content/dub22
!git clone --depth 1 https://github.com/FireRedTeam/FireRedTTS3.git /content/dub22/vendor/FireRedTTS3
!pip -q install -r /content/dub22/requirements-fireredtts3-colab.txt
print('تم تثبيت workspace وFireRedTTS3 مع الحفاظ على PyTorch الجاهز في Colab.')


In [ ]:
# POST_INSTALL_PREFLIGHT
import shutil, subprocess, torch
print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch فقد CUDA بعد التثبيت؛ لن أنزّل الأوزان.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM allocated/reserved (MiB):', round(torch.cuda.memory_allocated()/2**20), round(torch.cuda.memory_reserved()/2**20))
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], capture_output=True, text=True, check=True).stdout.strip())
usage = shutil.disk_usage('/content')
print('disk free GiB:', round(usage.free/2**30, 1), '/', round(usage.total/2**30, 1))


## تنزيل ملفات FireRedTTS3-Base فقط

نحمّل ملفات Base اللازمة للدبلجة ولا نحمّل نسخة Instruct غير المطلوبة.


In [ ]:
%cd /content/dub22/vendor/FireRedTTS3
!mkdir -p pretrained_models
!hf download FireRedTeam/FireRedTTS3 --local-dir pretrained_models --include 'redae/*' --include 'fireredtts3_base/*' --include 'campp/*' --include 'text_tokenizer/*'
from pathlib import Path
required = [
    Path('pretrained_models/redae/model.safetensors'),
    Path('pretrained_models/fireredtts3_base/model.safetensors'),
    Path('pretrained_models/campp/campplus_voxceleb.bin'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('ملفات النموذج المفقودة: ' + ', '.join(missing))
print('تم تنزيل ملفات FireRedTTS3-Base.')
%cd /content/dub22


## وضع T4 منخفض الذاكرة

إذا كانت البطاقة Tesla T4، تعدّل هذه الخلية FireRedTTS3 إلى float16 لأن T4 لا يناسبه مسار bfloat16 الافتراضي. لا تتجاوز هذه الخلية.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'
if 'T4' in gpu.stdout:
    print('T4 detected: enabling float16 compatibility mode')
    for source_file in [
        Path('/content/dub22/vendor/FireRedTTS3/fireredtts3/llm/fireredtts3_base.py'),
        Path('/content/dub22/vendor/FireRedTTS3/fireredtts3/redae/redae.py'),
    ]:
        text = source_file.read_text(encoding='utf-8')
        source_file.write_text(text.replace('torch.bfloat16', 'torch.float16'), encoding='utf-8')
    for config_file in Path('/content/dub22/vendor/FireRedTTS3/pretrained_models').rglob('config.json'):
        text = config_file.read_text(encoding='utf-8')
        config_file.write_text(text.replace('bfloat16', 'float16'), encoding='utf-8')
    print('تم تفعيل وضع T4 float16.')
else:
    print('GPU ليست T4؛ سيُستخدم إعداد FireRedTTS3 الافتراضي.')


## اختيار الفيديو

اترك `use_upload=False` لتجربة الفيديو الموجود في المستودع، أو اجعله `True` لرفع فيديو جديد.


## اختبار صوتي قصير قبل الدبلجة الكاملة
هذه الخلية تولّد أول مقطع فقط للتحقق من تشغيل النموذج على T4. لا تشغّل خلية الدبلجة الكاملة قبل فحص الصوت الناتج.


In [ ]:
# FIRED_SMOKE_TEST
import json, subprocess, sys
manifest_path = Path('/content/dub22/manifests/new_job/dialogue_ar_fireredtts3.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
smoke_manifest_path = Path('/content/dub22/manifests/new_job/dialogue_ar_fireredtts3_smoke.json')
smoke_manifest = dict(manifest)
smoke_manifest['segments'] = manifest['segments'][:1]
smoke_manifest_path.write_text(json.dumps(smoke_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
smoke_output = Path('/content/dub22/outputs/new_job/fireredtts3_smoke_line_01.mp4')
smoke_workdir = Path('/content/dub22/assets/fireredtts3/colab_smoke')
command = [
    sys.executable, '/content/dub22/scripts/firered_dub.py',
    '--input', str(input_video),
    '--manifest', str(smoke_manifest_path),
    '--source-root', '/content/dub22/vendor/FireRedTTS3',
    '--model-dir', '/content/dub22/vendor/FireRedTTS3/pretrained_models',
    '--output', str(smoke_output),
    '--workdir', str(smoke_workdir),
    '--duration', '59.4', '--timesteps', '8', '--cfg', '2.0',
]
print('بدء اختبار FireRedTTS3 للمقطع الأول فقط...')
result = subprocess.run(command, cwd='/content/dub22', text=True)
if result.returncode != 0 or not smoke_output.exists() or smoke_output.stat().st_size == 0:
    raise RuntimeError('فشل اختبار المقطع الواحد؛ لن أشغّل الدبلجة الكاملة.')
print('نجح الاختبار تقنيًا:', smoke_output)
print(subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(smoke_output)], capture_output=True, text=True, check=True).stdout.strip(), 'seconds')


In [ ]:
from pathlib import Path
from google.colab import files

default_video = Path('/content/dub22/assets/input/new_job/source.mp4')
use_upload = False
if use_upload:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('ارفع ملف فيديو واحدًا فقط.')
    name = next(iter(uploaded))
    input_video = Path('/content/dub22/assets/input/colab_job') / name
    input_video.parent.mkdir(parents=True, exist_ok=True)
    input_video.write_bytes(uploaded[name])
else:
    input_video = default_video
if not input_video.exists():
    raise FileNotFoundError(input_video)
print('الفيديو:', input_video)


## تشغيل الدبلجة

يستخدم هذا المثال manifest الفيديو الموجود في المستودع. عدّل `manifest_path` إذا أضفت فيديو ومخطط حوار جديدًا.


In [ ]:
import json
import subprocess
import sys

manifest_path = Path('/content/dub22/manifests/new_job/dialogue_ar_fireredtts3.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for segment in manifest['segments']:
    reference = Path('/content/dub22') / segment['reference_audio']
    if not reference.exists():
        raise FileNotFoundError(reference)

output_video = Path('/content/dub22/outputs/new_job/arabic_dub_fireredtts3.mp4')
output_video.parent.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '/content/dub22/scripts/firered_dub.py',
    '--input', str(input_video),
    '--manifest', str(manifest_path),
    '--source-root', '/content/dub22/vendor/FireRedTTS3',
    '--model-dir', '/content/dub22/vendor/FireRedTTS3/pretrained_models',
    '--output', str(output_video),
    '--workdir', '/content/dub22/assets/fireredtts3/colab_job',
    '--duration', '59.4',
    '--timesteps', '10',
    '--cfg', '2.0',
]
print('بدء FireRedTTS3...')
result = subprocess.run(command, cwd='/content/dub22', text=True)
if result.returncode != 0:
    raise RuntimeError('فشل FireRedTTS3؛ راجع آخر سطور الخلية وتحقق من GPU.')
if not output_video.exists() or output_video.stat().st_size == 0:
    raise RuntimeError('لم ينتج FireRedTTS3 ملف فيديو.')
print('تم الإنتاج:', output_video)


In [ ]:
probe = subprocess.run([
    'ffprobe', '-v', 'error',
    '-show_entries', 'format=duration:stream=codec_type,codec_name',
    '-of', 'default=noprint_wrappers=1', str(output_video),
], capture_output=True, text=True, check=True)
print(probe.stdout)
subprocess.run(['ffmpeg', '-v', 'error', '-i', str(output_video), '-f', 'null', '-'], check=True)
from IPython.display import Video, display
display(Video(str(output_video), embed=False))
files.download(str(output_video))
